### Tool

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code, Tools are pairings of:


    1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)
    2. A function or coroutine to execute

In [1]:
from langchain_ollama import ChatOllama

# ollama model
ollama_model = ChatOllama(model="qwen2.5:3b", temperature=0)
response = ollama_model.invoke("Why do parrots talk?")
response


AIMessage(content='Parrots are known for their ability to mimic human speech and sounds, a behavior that is often referred to as "talking." This ability is not as sophisticated as human speech, but it is a fascinating aspect of parrot behavior. Here are some reasons why parrots can talk:\n\n1. **Genetic and Neurological Factors**: Parrots have a unique brain structure that allows them to learn and mimic sounds. Their vocal cords and larynx are well-suited for producing a wide range of sounds, including human speech.\n\n2. **Social Learning**: Parrots are highly social animals and often learn vocalizations from their parents or other parrots. They can also learn from humans, especially if they are exposed to speech sounds early in their lives.\n\n3. **Environmental Factors**: The environment in which a parrot is raised can influence its ability to mimic sounds. For example, a parrot that is frequently exposed to human speech is more likely to learn to talk.\n\n4. **Morphology**: Parrots

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = ollama_model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What is the weather like in Tanzania")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Tanzania'}


### Tool Execution Loops


In [4]:
# Step 1: Model generates tool calls
messages = [{
    "role": "user",
    "content": "What is the weather in Tanzania?"
}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Tanzanis is 72F and sunny."

The weather in Tanzania is sunny.
